# Hypothetical Document Embeddings (HyDE) in Elasticsearch

HyDE rewrites a short user query as a fake document, embeds the fake document, and uses that vector to search. The fake document is thrown away; only real corpus documents are returned to the user.

Reference: [Precise Zero-Shot Dense Retrieval without Relevance Labels](https://arxiv.org/abs/2212.10496) (Gao et al., 2022).

## Setup

In [ ]:
%pip install -qU elasticsearch python-dotenv datasets matplotlib

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

ELASTICSEARCH_API_KEY = os.getenv("ELASTICSEARCH_API_KEY")
ELASTICSEARCH_URL = os.getenv("ELASTICSEARCH_URL")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

## Connect to Elasticsearch

In [ ]:
from elasticsearch import Elasticsearch

es_client = Elasticsearch(
    ELASTICSEARCH_URL,
    api_key=ELASTICSEARCH_API_KEY,
)
es_client.info()

## Load the dataset

We sample 5,000 papers from [`CShorten/ML-ArXiv-Papers`](https://huggingface.co/datasets/CShorten/ML-ArXiv-Papers): machine-learning arXiv titles and abstracts.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("CShorten/ML-ArXiv-Papers", split="train")
dataset = dataset.shuffle(seed=42).select(range(5000))
print(f"Sampled {len(dataset)} papers")
print(f"Columns: {dataset.column_names}")
dataset[0]

## Index with `semantic_text`

We use the `.jina-embeddings-v5-text-small` inference endpoint, preconfigured in Elastic Cloud. `copy_to` lets a single `semantic_text` field cover both `title` and `abstract`.

In [ ]:
INDEX_NAME = "arxiv-ml-papers"

if es_client.indices.exists(index=INDEX_NAME):
    es_client.indices.delete(index=INDEX_NAME)

es_client.indices.create(
    index=INDEX_NAME,
    mappings={
        "properties": {
            "title": {"type": "text", "copy_to": "semantic_content"},
            "abstract": {"type": "text", "copy_to": "semantic_content"},
            "semantic_content": {
                "type": "semantic_text",
                "inference_id": ".jina-embeddings-v5-text-small",
            },
        }
    },
)
print(f"Created index: {INDEX_NAME}")

In [ ]:
from elasticsearch import helpers


def build_bulk_actions(dataset, index_name):
    for i, item in enumerate(dataset):
        yield {
            "_index": index_name,
            "_id": i,
            "_source": {
                "title": item["title"],
                "abstract": item["abstract"],
            },
        }


success, failed = helpers.bulk(
    es_client,
    build_bulk_actions(dataset, INDEX_NAME),
    refresh=True,
)
print(f"Indexed {success} papers into '{INDEX_NAME}'")

## Create the chat completion inference endpoint

We register an OpenAI `gpt-4o-mini` endpoint via the Elasticsearch Inference API. Going through ES (instead of calling OpenAI directly) keeps API key handling, retries and observability inside Elasticsearch.

In [ ]:
from elasticsearch import NotFoundError

HYDE_INFERENCE_ID = "hyde-completion"

# idempotent: remove a stale endpoint from a previous run
try:
    es_client.inference.delete(inference_id=HYDE_INFERENCE_ID)
except NotFoundError:
    pass

response = es_client.inference.put(
    task_type="completion",
    inference_id=HYDE_INFERENCE_ID,
    inference_config={
        "service": "openai",
        "service_settings": {
            "api_key": OPENAI_API_KEY,
            "model_id": "gpt-4o-mini",
        },
    },
)
print(f"Created inference endpoint: {HYDE_INFERENCE_ID}")
response

## Baseline search

In [ ]:
def search(query_text, size=5):
    response = es_client.search(
        index=INDEX_NAME,
        query={
            "semantic": {
                "field": "semantic_content",
                "query": query_text,
            }
        },
        size=size,
        _source=["title", "abstract"],
    )
    return response["hits"]["hits"]


def print_hits(hits):
    for i, hit in enumerate(hits, 1):
        print(f"{i}. [{hit['_score']:.3f}] {hit['_source']['title']}")


query = "why does adamw train transformers better than plain adam"
baseline_hits = search(query)
print(f"Query: {query}\n")
print("Top 5 (raw query):")
print_hits(baseline_hits)

## Generate the hypothetical document

We ask the LLM for a 150-200 word abstract in formal academic register, low temperature so generations are near-deterministic and cacheable by query text.

In [ ]:
def generate_hypothetical_abstract(query):
    prompt = (
        "You are helping improve search over a corpus of machine learning "
        "paper abstracts from arXiv.\n\n"
        "Given a short user query, write ONE plausible research paper abstract "
        "(150-200 words) that would be a perfect match for that query. Use the "
        "formal register and density of a real arXiv abstract: methods, setup, "
        "findings. Do not add a title, headings, or any explanation. Return "
        "only the abstract text itself.\n\n"
        f"User query: {query}\n\nHypothetical abstract:"
    )

    response = es_client.inference.completion(
        inference_id=HYDE_INFERENCE_ID,
        input=prompt,
    )
    return response["completion"][0]["result"].strip()


hypothetical = generate_hypothetical_abstract(query)
print(hypothetical)

## Evaluate HyDE vs baseline against a judgment set

For each query we hand-pick the titles in the corpus we consider relevant, then run Elasticsearch's [`_rank_eval`](https://www.elastic.co/docs/api/doc/elasticsearch/operation/operation-rank-eval) API to score baseline vs HyDE on:

- **Precision@5**: fraction of the top-5 that are in the relevance set.
- **Recall@5**: fraction of the relevance set captured in the top-5.

Adjust the title sets below if your run surfaces different candidates you'd consider correct.

In [ ]:
RELEVANT_DOCS = {
    "why does adamw train transformers better than plain adam": {
        "Understanding AdamW through Proximal Methods and Scale-Freeness",
        "Maximizing Communication Efficiency for Large-scale Training via 0/1 Adam",
        "Subformer: Exploring Weight Sharing for Parameter Efficiency in Generative Transformers",
        "Train Large, Then Compress: Rethinking Model Size for Efficient Training and Inference of Transformers",
        "Trainable Weight Averaging for Fast Convergence and Better Generalization",
    },
    "is mixture of experts worth it for small language models": {
        "Task-Specific Expert Pruning for Sparse Mixture-of-Experts",
        "Exploring Extreme Parameter Compression for Pre-trained Language Models",
        "Train Large, Then Compress: Rethinking Model Size for Efficient Training and Inference of Transformers",
        "Balancing Expert Utilization in Mixture-of-Experts Layers Embedded in CNNs",
        "Deep Ensembles on a Fixed Memory Budget: One Wide Network or Several Thinner Ones?",
        "Learning Factored Representations in a Deep Mixture of Experts",
    },
    "how do you stop catastrophic forgetting during fine-tuning": {
        "Understanding the Role of Training Regimes in Continual Learning",
        "SupportNet: solving catastrophic forgetting in class incremental learning with support data",
        "Explain to Not Forget: Defending Against Catastrophic Forgetting with XAI",
        "Few-shot Continual Learning: a Brain-inspired Approach",
        "Online Continual Learning under Extreme Memory Constraints",
        "Continual Learning in Deep Neural Network by Using a Kalman Optimiser",
        "On Tiny Episodic Memories in Continual Learning",
    },
}

In [ ]:
import re
import pandas as pd
from IPython.display import display


def normalize(title):
    return re.sub(r"\s+", " ", title).strip()


# Map (normalized) titles to the doc _ids we used during bulk indexing
title_to_id = {normalize(item["title"]): str(i) for i, item in enumerate(dataset)}
id_to_title = {str(i): item["title"] for i, item in enumerate(dataset)}


def to_ratings(relevant_titles):
    return [
        {"_index": INDEX_NAME, "_id": title_to_id[normalize(t)], "rating": 1}
        for t in relevant_titles
        if normalize(t) in title_to_id
    ]


def build_request(query_id, query_text, relevant_titles, k=5):
    return {
        "id": query_id,
        "request": {
            "query": {"semantic": {"field": "semantic_content", "query": query_text}},
            "size": k,
        },
        "ratings": to_ratings(relevant_titles),
    }


queries = list(RELEVANT_DOCS.keys())

# Generate each hypothetical once and reuse it across metric runs
hypos = {q: generate_hypothetical_abstract(q) for q in queries}

baseline_requests = [
    build_request(f"q{i}", q, RELEVANT_DOCS[q]) for i, q in enumerate(queries)
]
hyde_requests = [
    build_request(f"q{i}", hypos[q], RELEVANT_DOCS[q]) for i, q in enumerate(queries)
]


METRICS = {
    "precision@5": {"precision": {"k": 5, "relevant_rating_threshold": 1}},
    "recall@5": {"recall": {"k": 5, "relevant_rating_threshold": 1}},
}


eval_responses = {}
rows = []
for method, requests in [("baseline", baseline_requests), ("hyde", hyde_requests)]:
    per_query = {q: {"query": q, "method": method} for q in queries}
    for metric_name, metric_body in METRICS.items():
        response = es_client.rank_eval(
            index=INDEX_NAME, requests=requests, metric=metric_body
        )
        for i, q in enumerate(queries):
            per_query[q][metric_name] = response["details"][f"q{i}"]["metric_score"]
        if metric_name == "precision@5":
            eval_responses[method] = response["details"]
    rows.extend(per_query.values())


scores = pd.DataFrame(rows).set_index(["query", "method"])
display(scores)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

BASELINE_COLOR = "#4C72B0"
HYDE_COLOR = "#DD8452"

queries = list(RELEVANT_DOCS.keys())
metric_names = ["precision@5", "recall@5"]

fig, axes = plt.subplots(2, 1, figsize=(9, 8))
ys = np.arange(len(queries))
height = 0.38

for ax, metric in zip(axes, metric_names):
    baseline_vals = [scores.loc[(q, "baseline"), metric] for q in queries]
    hyde_vals = [scores.loc[(q, "hyde"), metric] for q in queries]
    ax.barh(
        ys + height / 2, baseline_vals, height, label="baseline", color=BASELINE_COLOR
    )
    ax.barh(ys - height / 2, hyde_vals, height, label="hyde", color=HYDE_COLOR)
    ax.set_yticks(ys)
    ax.set_yticklabels(queries, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(metric)
    ax.set_xlim(0, 1)
    ax.set_xlabel("score")
    ax.grid(axis="x", alpha=0.3)

axes[0].legend(loc="lower right")
fig.suptitle("HyDE vs baseline on a hand-curated judgment set")
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side diagnostic: which titles each method retrieves
for r in results:
    relevant_norm = {normalize(t) for t in r["relevant"]}
    print(f"\nQuery: {r['query']}")
    print("Baseline:")
    for h in r["baseline_hits"]:
        title = h["_source"]["title"]
        mark = "✓" if normalize(title) in relevant_norm else " "
        print(f"  {mark} [{h['_score']:.3f}] {title}")
    print("HyDE:")
    for h in r["hyde_hits"]:
        title = h["_source"]["title"]
        mark = "✓" if normalize(title) in relevant_norm else " "
        print(f"  {mark} [{h['_score']:.3f}] {title}")

In [ ]:
es_client.indices.delete(index=INDEX_NAME)
print(f"Deleted index: {INDEX_NAME}")

es_client.inference.delete(inference_id=HYDE_INFERENCE_ID)
print(f"Deleted inference endpoint: {HYDE_INFERENCE_ID}")